<a href="https://colab.research.google.com/github/rudrakj/Assist/blob/master/Youtube_NLP_Text_Representaion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install sklearn -q
# !pip install spacy -q
# !python -m spacy download en_core_web_sm

In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn import metrics

import string
import spacy
np.random.seed(42)

In [2]:
data = pd.read_csv("train.csv", engine="python")
data.head()

,comment_text,toxic
0,Explanation\nWhy the edits made under my usern...,0.0
1,D'aww! He matches this background colour I'm s...,0.0
2,"Hey man, I'm really not trying to edit war. It...",0.0
3,"""\nMore\nI can't make any real suggestions on ...",0.0
4,"You, sir, are my hero. Any chance you remember...",0.0


In [3]:
nlp = spacy.load("en_core_web_sm")
stop_words = nlp.Defaults.stop_words
print(stop_words)

{'formerly', 'forty', 'during', 'whither', 'a', '’re', 'done', 'it', 'nevertheless', 'seemed', 'itself', 'most', 'him', 'above', 'nine', 'so', 'fifteen', 'me', 'into', 'around', 'first', 'through', 'some', 'we', 'whatever', 'where', 'hence', 'nobody', 'such', 'who', 'to', 'they', 'everyone', 'whereupon', 'below', 'further', 'show', 'namely', 'ca', 'throughout', 'something', 'too', 'indeed', 'than', 'in', 'therein', 'now', 'among', 'former', 'serious', 'is', 'her', 'latterly', 'mine', 'about', 'whereafter', 'must', 'that', 'himself', 'only', 'become', 'alone', 'put', 'how', 'being', '‘s', 'own', 'might', 'has', 'eleven', 'cannot', 'whereas', 'thereupon', 'any', 'quite', 'however', 'when', 'bottom', 'your', 'eight', "'ve", 'whole', 'next', 'much', 'by', 'every', 'amongst', 'not', 'with', 'using', 'beyond', 'always', 'herein', 'latter', '’ll', 'or', 'front', 'n‘t', 'three', 'seem', 'per', 'did', 'do', 'she', 'themselves', 'hers', 'along', 'he', 'though', 'under', 'whether', 'often', "'ll"

In [4]:
punctuations = string.punctuation
print(punctuations)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [5]:
# Creating our tokenizer function
def spacy_tokenizer(sentence):
    # Creating our token object, which is used to create documents with linguistic annotations.
    doc = nlp(sentence)



    # print(doc)
    # print(type(doc))

    # Lemmatizing each token and converting each token into lowercase
    mytokens = [ word.lemma_.lower().strip() for word in doc ]

    # print(mytokens)

    # Removing stop words
    mytokens = [ word for word in mytokens if word not in stop_words and word not in punctuations ]

    # return preprocessed list of tokens
    return mytokens

In [6]:
sentence = "I am eating apple ?"
spacy_tokenizer(sentence)

['eat', 'apple']

In [7]:
count_vector = CountVectorizer(tokenizer = spacy_tokenizer)
# tfidf_vector = TfidfVectorizer(tokenizer = spacy_tokenizer)

In [8]:
count_vector.fit_transform(["I am eating apple, I like apple","I am playing cricket"]).toarray()

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


array([[2, 0, 1, 1, 0],
       [0, 1, 0, 0, 1]])

In [9]:
count_vector.get_feature_names_out()

array(['apple', 'cricket', 'eat', 'like', 'play'], dtype=object)

In [10]:
count_vector.vocabulary_

{'eat': 2, 'apple': 0, 'like': 3, 'play': 4, 'cricket': 1}

In [12]:
data[data['comment_text'].isna()]

,comment_text,toxic
499,NaN,NaN
500,NaN,NaN
501,NaN,NaN
502,NaN,NaN
503,NaN,NaN
...,...,...
159566,NaN,NaN
159567,NaN,NaN
159568,NaN,NaN
159569,NaN,NaN


In [13]:
data.dropna(inplace=True)

In [14]:
data.duplicated().sum()

np.int64(0)

In [15]:
data.drop_duplicates(inplace=True)

In [16]:
from sklearn.model_selection import train_test_split

X = data['comment_text'] # the features we want to analyze
ylabels = data['toxic'] # the labels, or answers, we want to test against

X_train, X_test, y_train, y_test = train_test_split(X, ylabels, test_size=0.2,stratify=ylabels)

In [ ]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression()

In [ ]:
X_train_vetcors= count_vector.fit_transform(X_train)
X_test_vetcors= count_vector.transform(X_test)

In [ ]:
X_train_vetcors.shape

(800, 5783)

In [ ]:
X_test_vetcors.shape

(200, 5783)

In [ ]:
X_train_vetcors.toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
classifier.fit(X_train_vetcors,y_train)

LogisticRegression()

In [ ]:
predicted = classifier.predict(X_test_vetcors)
print("Logistic Regression Accuracy:",metrics.accuracy_score(y_test, predicted))
print("Logistic Regression Precision:",metrics.precision_score(y_test, predicted))
print("Logistic Regression Recall:",metrics.recall_score(y_test, predicted))

Logistic Regression Accuracy: 0.845
Logistic Regression Precision: 0.8709677419354839
Logistic Regression Recall: 0.81


In [ ]:
tfidf_vector = TfidfVectorizer(tokenizer = spacy_tokenizer)
X_train_vetcors= tfidf_vector.fit_transform(X_train)
X_test_vetcors= tfidf_vector.transform(X_test)

In [ ]:
classifier = LogisticRegression()
classifier.fit(X_train_vetcors,y_train)
predicted = classifier.predict(X_test_vetcors)
print("Logistic Regression Accuracy:",metrics.accuracy_score(y_test, predicted))
print("Logistic Regression Precision:",metrics.precision_score(y_test, predicted))
print("Logistic Regression Recall:",metrics.recall_score(y_test, predicted))

Logistic Regression Accuracy: 0.86
Logistic Regression Precision: 0.9090909090909091
Logistic Regression Recall: 0.8
